In [ ]:
# !pip install requests
# !pip install bs4

In [36]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_cities_and_towns_in_Trinidad_and_Tobago"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# Find ALL tables
tables = soup.find_all("table", {"class": "wikitable"})

data = []

for table in tables:
    rows = table.find_all("tr")

    for row in rows[1:]:  # skip header
        cols = row.find_all("td")
        cols = [col.text.strip() for col in cols]

        if len(cols) >= 2:
            data.append(cols[:2])  # City/Town + Region

# Create dataframe
raw_data = pd.DataFrame(data, columns=["City/Town", "Region"])

display(raw_data.head())

,City/Town,Region
0,Aranguez,San Juan–Laventille
1,Arima,Borough of Arima
2,Arena,Couva-Tabaquite-Talparo
3,Arnos Vale,Tobago
4,Arouca,Tunapuna–Piarco


In [34]:
import pandas as pd

# Example DataFrame
data = {
    "City/Town": [
        "Canaan, Tobago",
        "Canaan, Trinidad",
        "Dow Village, Couva",
        "Dow Village, Siparia",
        "Las Lomas, Trinidad and Tobago",
        "Lambeau, Tobago",
        "Lengua, Trinidad and Tobago",
        "Arquart (Aquat, Aquart, or Urquart) Village"
    ]
}

test_df = pd.DataFrame(data)

# Cleaning function
import pandas as pd
import re

def clean_city_town(x):
    if pd.isna(x):
        return x

    # 1️⃣ Remove anything inside parentheses
    x_no_parens = re.sub(r"\(.*?\)", "", x).strip()

    # 2️⃣ Apply Trinidad/Tobago logic
    if "Trinidad" in x or "Tobago" in x:
        # Keep everything before first comma
        result = x_no_parens.split(",", 1)[0].strip()
    elif "," in x_no_parens:
        # Keep everything before and after comma, drop the comma
        result = x_no_parens.replace(",", "").strip()
    else:
        result = x_no_parens.strip()

    # 3️⃣ Replace multiple spaces with a single space
    result = re.sub(r"\s+", " ", result)

    return result

# Apply to the column
test_df["City/Town_Clean"] = test_df["City/Town"].apply(clean_city_town)

print(test_df)

                                     City/Town      City/Town_Clean
0                               Canaan, Tobago               Canaan
1                             Canaan, Trinidad               Canaan
2                           Dow Village, Couva    Dow Village Couva
3                         Dow Village, Siparia  Dow Village Siparia
4               Las Lomas, Trinidad and Tobago            Las Lomas
5                              Lambeau, Tobago              Lambeau
6                  Lengua, Trinidad and Tobago               Lengua
7  Arquart (Aquat, Aquart, or Urquart) Village      Arquart Village


In [40]:
# Clean data
df = raw_data.copy()
# df = df.dropna()
df["City/Town"].isna().sum()

df["City/Town"] = df["City/Town"].apply(clean_city_town)

df = df[~(df["City/Town"].isna() | (df["City/Town"].str.strip() == ""))]

df = df.reset_index(drop=True)

df["city_id"] = range(1, len(df) + 1)

# Move 'city_id' to the front
cols = df.columns.tolist()
cols.insert(0, cols.pop(cols.index('city_id')))
df = df[cols]

display(df.head())
display(df.tail())

,city_id,City/Town,Region
0,1,Aranguez,San Juan–Laventille
1,2,Arima,Borough of Arima
2,3,Arena,Couva-Tabaquite-Talparo
3,4,Arnos Vale,Tobago
4,5,Arouca,Tunapuna–Piarco


,city_id,City/Town,Region
279,280,Waterloo,Couva–Tabaquite–Talparo
280,281,Westmoorings,Diego Martin region
281,282,Whiteland,Couva–Tabaquite–Talparo
282,283,Williamsville,Princes Town region
283,284,Woodbrook,Port of Spain


In [42]:
# Save file
file_name = "trinidad_and_tobago_all_cities_towns.csv"

df.to_csv(file_name, index=False)

print("Saved to " + file_name)

Saved to trinidad_and_tobago_all_cities_towns.csv
